In [2]:
!pip install transformers torch seqeval


import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification
from sklearn.metrics import classification_report
from seqeval.metrics import precision_score, recall_score, f1_score



# TASK 1 — DATASET 


# Labels:
# 0 = O
# 1 = B-PER
# 2 = I-PER
# 3 = B-ORG
# 4 = I-ORG
# 5 = B-LOC
# 6 = I-LOC

label_list = ["O","B-PER","I-PER","B-ORG","I-ORG","B-LOC","I-LOC"]

label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for i,l in enumerate(label_list)}

# Sample dataset
data = [
    (["John","works","at","Google"], [1,0,0,3]),
    (["Mary","lives","in","London"], [1,0,0,5]),
    (["Alice","is","from","Paris"], [1,0,0,5]),
    (["Bob","joined","Microsoft"], [1,0,3]),
    (["Amazon","is","a","company"], [3,0,0,0]),
]


#TASK-2

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

class TokenDataset(Dataset):
    def __init__(self, data):
        self.encodings = []
        self.labels = []

        for tokens, tags in data:
            enc = tokenizer(tokens, is_split_into_words=True, truncation=True, padding='max_length', max_length=10)

            word_ids = enc.word_ids()
            label_ids = []
            previous_word = None

            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word:
                    label_ids.append(tags[word_idx])
                else:
                    label_ids.append(-100)
                previous_word = word_idx

            self.encodings.append(enc)
            self.labels.append(label_ids)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v) for k,v in self.encodings[idx].items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


dataset = TokenDataset(data)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# TASK 3 — MODEL

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# TASK 4 — TRAINING

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

model.train()

for epoch in range(2):
    print(f"\nEpoch {epoch+1}")
    for batch in loader:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

    print("Loss:", loss.item())

# TASK 5 — EVALUATION

model.eval()

true_labels = []
pred_labels = []

with torch.no_grad():
    for batch in loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=2)

        for pred, lab in zip(predictions, labels):
            temp_pred = []
            temp_lab = []

            for p, l in zip(pred, lab):
                if l != -100:
                    temp_pred.append(id2label[p.item()])
                    temp_lab.append(id2label[l.item()])

            pred_labels.append(temp_pred)
            true_labels.append(temp_lab)

print("\nPrecision:", precision_score(true_labels, pred_labels))
print("Recall:", recall_score(true_labels, pred_labels))
print("F1 Score:", f1_score(true_labels, pred_labels))

# TASK 6 — INFERENCE

def predict(sentence):
    tokens = sentence.split()

    enc = tokenizer(tokens, is_split_into_words=True, return_tensors="pt")
    enc = {k:v.to(device) for k,v in enc.items()}

    outputs = model(**enc)
    preds = torch.argmax(outputs.logits, dim=2)[0]

    word_ids = enc["input_ids"][0]

    results = []

    for i, token in enumerate(tokens):
        label = id2label[preds[i+1].item()]  # skip [CLS]
        results.append((token, label))

    return results


print("\nInference:")
print(predict("John works at Google in California"))

Loading weights: 100%|█████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 1989.45it/s]
BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading 


Epoch 1
Loss: 1.7049338817596436

Epoch 2
Loss: 1.5217523574829102

Precision: 1.0
Recall: 0.2222222222222222
F1 Score: 0.3636363636363636

Inference:
[('John', 'O'), ('works', 'O'), ('at', 'O'), ('Google', 'O'), ('in', 'O'), ('California', 'O')]


Task 7: Comparison — POS Tagging vs Chunking

POS Tagging and Chunking are both important NLP tasks, but they work at different levels.

POS Tagging:
POS (Part-of-Speech) tagging assigns a grammatical label to each individual word in a sentence.
Examples include noun, verb, adjective, etc.

Example:
Sentence: "John works at Google"
John → Proper Noun
works → Verb
Google → Proper Noun

Chunking:
Chunking groups words together into meaningful phrases such as noun phrases or verb phrases.

Example:
"John works at Google"
[John] → Noun Phrase
[works] → Verb Phrase
[at Google] → Prepositional Phrase

Key Differences:
- POS Tagging works at word level
- Chunking works at phrase level
- POS tagging is simpler and more basic
- Chunking gives more contextual structure

Conclusion:
POS tagging helps understand grammar, while chunking helps understand sentence structure.

Task 8: Report — BERT Token Classification

In this project, I implemented a token classification model using BERT for POS tagging and chunking tasks.

The goal was to understand how transformer models can be used for sequence labeling tasks in NLP.

Key Steps:
- Loaded and prepared dataset
- Applied tokenization using BERT tokenizer
- Handled label alignment with subwords using -100
- Trained BERT model for token classification
- Evaluated using Precision, Recall, and F1 Score
- Performed inference on custom sentences

Challenges Faced:
- Handling subword tokenization was difficult
- Aligning labels correctly with tokens required careful logic
- Managing special tokens like [CLS] and [SEP]

Observations:
- BERT performs very well for token classification tasks
- Proper preprocessing and label alignment are critical
- Transformer models capture contextual meaning better than traditional methods

Conclusion:
This project helped me understand how modern NLP models like BERT can be fine-tuned for real-world applications such as POS tagging and chunking.